# Variational optimization loop

Run the same fixed-step gradient-descent loop with a reference QNode and the MettleQ device.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [2]:
def make_qnode(device):
    @qml.qnode(device, diff_method="parameter-shift")
    def circuit(weights):
        qml.RY(weights[0], wires=0)
        qml.RX(weights[1], wires=1)
        qml.CNOT(wires=[0, 1])
        qml.RY(weights[2], wires=1)
        return qml.expval(qml.Z(0) @ qml.Z(1))
    return circuit

def train(qnode):
    weights = pnp.array([0.2, -0.4, 0.7], requires_grad=True)
    trace = []
    for _ in range(8):
        value = qnode(weights)
        trace.append(float(value))
        weights = weights - 0.15 * qml.grad(qnode)(weights)
    trace.append(float(qnode(weights)))
    return np.asarray(trace)

reference_qnode = make_qnode(qml.device("default.qubit", wires=2))
reference, reference_ms, _ = benchmark(lambda: train(reference_qnode), repeats=2)
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: train(mettleq_qnode), repeats=2)
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/04_variational_optimization.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="optimization trace atol=4e-5",
    passed=error <= 4e-5 and candidate[-1] <= candidate[0],
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_trace_error": error, "reference_trace": reference, "mettleq_trace": candidate},
)

TUTORIAL_RESULT::{"check": "optimization trace atol=4e-5", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"max_trace_error": 9.725507865709915e-08, "mettleq_trace": [0.7044664025306702, 0.6360342502593994, 0.5586625933647156, 0.47383859753608704, 0.38371843099594116, 0.2907371520996094, 0.19708256423473358, 0.10422448813915253, 0.012694669887423515], "reference_trace": [0.7044663052755915, 0.6360342844556788, 0.5586625801001408, 0.4738386056809144, 0.38371844500894703, 0.2907372202985556, 0.19708257496945103, 0.10422440007088346, 0.012694577493989834]}, "mettleq_median_ms": 50.576562993228436, "notebook": "pennylane/04_variational_optimization.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 24.82808350760024, "reference_over_mettleq": 0.49090096357327423, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}
